In [1]:
#SOURCES:

#The dataset was downloaded from https://www.kaggle.com/datasets/sanjivpradhanang/ufc-fight-stats,
#The csv dataset was created by SANJIV PRADHANANG 
# All stats orginate from http://ufcstats.com

In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("ufc_fight_stats.csv")

In [4]:
df.head()

,fighter_1,fighter_2,knockdowns_1,knockdowns_2,total_strikes_1,total_strikes_2,significant_strikes_1,significant_strikes_2,head_strikes_1,head_strikes_2,...,reversals_1,reversals_2,control_time_1,control_time_2,result,method,round,time,referee,ufc_stats_com_url
0,Alexa Grasso,Valentina Shevchenko,1,0,262 of 408,199 of 301,84 of 203,80 of 179,54 of 151,61 of 158,...,1,0,2:39,8:37,D,Decision - Split,5,5:00,Herb Dean,http://ufcstats.com/fight-details/b395c89e19a3...
1,Kevin Holland,Jack Della Maddalena,0,0,127 of 356,105 of 190,127 of 356,105 of 190,59 of 258,46 of 115,...,0,0,0:00,0:00,L,Decision - Split,3,5:00,Jason Herzog,http://ufcstats.com/fight-details/697efaf0d162...
2,Raul Rosas Jr.,Terrence Mitchell,1,0,19 of 31,6 of 19,18 of 30,6 of 19,17 of 29,5 of 18,...,0,0,0:17,0:00,W,KO/TKO,1,0:54,Mark Smith,http://ufcstats.com/fight-details/c0ca4c201d08...
3,Daniel Zellhuber,Christos Giagos,0,0,36 of 101,38 of 82,36 of 101,38 of 82,18 of 75,25 of 63,...,0,0,0:17,0:11,W,Submission,2,3:26,Jason Herzog,http://ufcstats.com/fight-details/2e1435c160bf...
4,Fernando Padilla,Kyle Nelson,0,0,73 of 209,83 of 185,72 of 208,82 of 184,42 of 168,43 of 143,...,0,0,0:07,0:00,L,Decision - Unanimous,3,5:00,Chris Tognoni,http://ufcstats.com/fight-details/a5d76e93a505...


In [5]:
df.shape

(3561, 34)

In [6]:
for col in df.columns:
    print(col)
    
# Here is the titles of the 34 colums of this dataset. 

# Some non-numerical values can be discarded like : 'fighter_1'(name), 'fighter_2'(name), 'referee'(name) and
#'ufc_stats_com_url'(link from which data was extracted). Since they are not performance variables.

# On the other hand, some non-numerical values are very important like : 'result'(fight result : Win, loss, etc.) and 
#'method'(The method by which the result was obtained: Was the fight result a KO/TKO (automatic win)? 
# Was the fight result decided by the judges (non-KO/TKO result)? And was that judge decision unanimous or split?

# The numerical variables are easier to analyse. They are represented by simple numbers(EX:'knockdowns_1'), 
# a proportion (EX: 'total_strikes_1') that describe the number of strike landed / total number of strike attempted, 
# (EX: when total strikes are shown as 262 of 408 (first row), it means 262 strikes landed of 408 attempted) or,
# time (EX: 'time' that describe the time count of the last round of the fight).


fighter_1
fighter_2
knockdowns_1
knockdowns_2
total_strikes_1
total_strikes_2
significant_strikes_1
significant_strikes_2
head_strikes_1
head_strikes_2
body_strikes_1
body_strikes_2
leg_strikes_1
leg_strikes_2
distance_strikes_1
distance_strikes_2
clinch_strikes_1
clinch_strikes_2
ground_strikes_1
ground_strikes_2
takedowns_1
takedowns_2
submission_attempts_1
submission_attempts_2
reversals_1
reversals_2
control_time_1
control_time_2
result
method
round
time
referee
ufc_stats_com_url


In [7]:
del df['fighter_1']
del df['fighter_2']
del df['referee']
del df['ufc_stats_com_url']
df.head()

,knockdowns_1,knockdowns_2,total_strikes_1,total_strikes_2,significant_strikes_1,significant_strikes_2,head_strikes_1,head_strikes_2,body_strikes_1,body_strikes_2,...,submission_attempts_1,submission_attempts_2,reversals_1,reversals_2,control_time_1,control_time_2,result,method,round,time
0,1,0,262 of 408,199 of 301,84 of 203,80 of 179,54 of 151,61 of 158,12 of 22,6 of 8,...,1,1,1,0,2:39,8:37,D,Decision - Split,5,5:00
1,0,0,127 of 356,105 of 190,127 of 356,105 of 190,59 of 258,46 of 115,33 of 55,36 of 49,...,0,0,0,0,0:00,0:00,L,Decision - Split,3,5:00
2,1,0,19 of 31,6 of 19,18 of 30,6 of 19,17 of 29,5 of 18,1 of 1,1 of 1,...,0,0,0,0,0:17,0:00,W,KO/TKO,1,0:54
3,0,0,36 of 101,38 of 82,36 of 101,38 of 82,18 of 75,25 of 63,11 of 19,3 of 7,...,1,0,0,0,0:17,0:11,W,Submission,2,3:26
4,0,0,73 of 209,83 of 185,72 of 208,82 of 184,42 of 168,43 of 143,17 of 22,19 of 20,...,0,0,0,0,0:07,0:00,L,Decision - Unanimous,3,5:00


In [8]:
#HYPOTSIS :

# The main focus of this analysis is to find if there is any correlation between fighting styles and win rate.


# The target variable is the fight result ('result').


# I think that the principal components to predict the fight outcome are :

#    - (Landed)'Significant strikes' amount because although all strikes are important, significant strikes are
# the ones where the fighter really commits to it and inflicts the most damage when landed. The other type of strikes
# are sometimes thrown to set up another strike or to make the opponent flinch and distract him.

#    - Accuracy percentage of significant strikes (can be calculated by dividing the # of significant strikes landed by
# # of significant strikes attempted). Accuracy plays a major role in energy conservation of the fighter. If the fighter spends
# too much energy on strikes and doesn't land them, he will get tired and be at a disadvantage. Also, it's useful to calculate
# the accuracy of significant strikes but not so much for the other strikes since, like I mentioned before, some strikes are
# only thrown to make the opponent react a certain way to set up something else. In other words, fighter purposely miss some
# strikes the difference is, with significant strikes when you miss them, you lose a lot of energy which will make a difference 
# in the fight.
 
#    -  Control time. Control time is described by UFC Stats officials as the time a certain fighter has spendent in a dominant
# position over his opponent. This is crucial in the outcome of a fight since in a dominant position, a fight will usually spend
# less energy than his opponent and have more opportunities to land strikes.

In [9]:
# Replace fight result values by numerical values so it's easier to analyse (EX : correlation)
df['result'].replace({'W': 3, 'L': 1, 'D': 2, 'NC': 4}, inplace=True) 
df.head()

,knockdowns_1,knockdowns_2,total_strikes_1,total_strikes_2,significant_strikes_1,significant_strikes_2,head_strikes_1,head_strikes_2,body_strikes_1,body_strikes_2,...,submission_attempts_1,submission_attempts_2,reversals_1,reversals_2,control_time_1,control_time_2,result,method,round,time
0,1,0,262 of 408,199 of 301,84 of 203,80 of 179,54 of 151,61 of 158,12 of 22,6 of 8,...,1,1,1,0,2:39,8:37,2,Decision - Split,5,5:00
1,0,0,127 of 356,105 of 190,127 of 356,105 of 190,59 of 258,46 of 115,33 of 55,36 of 49,...,0,0,0,0,0:00,0:00,1,Decision - Split,3,5:00
2,1,0,19 of 31,6 of 19,18 of 30,6 of 19,17 of 29,5 of 18,1 of 1,1 of 1,...,0,0,0,0,0:17,0:00,3,KO/TKO,1,0:54
3,0,0,36 of 101,38 of 82,36 of 101,38 of 82,18 of 75,25 of 63,11 of 19,3 of 7,...,1,0,0,0,0:17,0:11,3,Submission,2,3:26
4,0,0,73 of 209,83 of 185,72 of 208,82 of 184,42 of 168,43 of 143,17 of 22,19 of 20,...,0,0,0,0,0:07,0:00,1,Decision - Unanimous,3,5:00


In [10]:
# 'No contest' result is obtained when a fighter land and illegal blow that causes his opponent to be injured and unable to 
# continue fighting (EX : eye poke). In a no contest fight, there is no winner. This means that fights that ends with a no 
# contest are useless this study since we cannot calculate winrate.

# Removing no contest fights from dataframe: 

df = df.loc[df['result'] < 4]


if '4' in df['result'] :
    print('''Value "4" is still in the column "result"''')
else :
    print('''Value "4" is not found in the column "result"''')
 

Value "4" is not found in the column "result"


In [11]:
# Verifying that dataframe was updated.
df.shape

(3524, 30)

In [12]:
# Separting strike and takedowns columns values into 2 columns each:

# Columns that count strikes and takedowns are described as a proportion (EX : in the first row, 'total_strikes_1' are 
# described as 262 of 408). It represents the strikes that landed vs the total strikes thrown or attempted by a fighter.
# For this reason, it's important to separte those data into 2 columns to be able to anlysis them since right now, they're 
# strings and not floats or integers.


    # Creating a list of the names of all columns:
    
df_StrikeList = []

for col in df.columns:
    df_StrikeList.append(f'{col}')
    
print(df_StrikeList)

['knockdowns_1', 'knockdowns_2', 'total_strikes_1', 'total_strikes_2', 'significant_strikes_1', 'significant_strikes_2', 'head_strikes_1', 'head_strikes_2', 'body_strikes_1', 'body_strikes_2', 'leg_strikes_1', 'leg_strikes_2', 'distance_strikes_1', 'distance_strikes_2', 'clinch_strikes_1', 'clinch_strikes_2', 'ground_strikes_1', 'ground_strikes_2', 'takedowns_1', 'takedowns_2', 'submission_attempts_1', 'submission_attempts_2', 'reversals_1', 'reversals_2', 'control_time_1', 'control_time_2', 'result', 'method', 'round', 'time']


In [27]:
    # Printing the names of columns that describe strikes and takedowns:
    
for ColName in df_StrikeList[2:20]:
    print(ColName)

IndentationError: expected an indented block after 'for' statement on line 3 (2461890586.py, line 4)

In [26]:
    # Separating columns that describe strikes and takedowns and creating new columns to input the separted values:

for ColName in df_StrikeList[2:20]:
        df[[f'{ColName} (landed)', f'{ColName} (attempted)']] = df[f'{ColName}'].str.split('of', expand=True)
df.head()

,knockdowns_1,knockdowns_2,total_strikes_1,total_strikes_2,significant_strikes_1,significant_strikes_2,head_strikes_1,head_strikes_2,body_strikes_1,body_strikes_2,...,clinch_strikes_2 (landed),clinch_strikes_2 (attempted),ground_strikes_1 (landed),ground_strikes_1 (attempted),ground_strikes_2 (landed),ground_strikes_2 (attempted),takedowns_1 (landed),takedowns_1 (attempted),takedowns_2 (landed),takedowns_2 (attempted)
0,1,0,262 of 408,199 of 301,84 of 203,80 of 179,54 of 151,61 of 158,12 of 22,6 of 8,...,6,8,2,8,7,13,1,2,4,7
1,0,0,127 of 356,105 of 190,127 of 356,105 of 190,59 of 258,46 of 115,33 of 55,36 of 49,...,11,15,0,0,0,0,0,2,0,0
2,1,0,19 of 31,6 of 19,18 of 30,6 of 19,17 of 29,5 of 18,1 of 1,1 of 1,...,0,0,11,17,0,0,0,0,0,0
3,0,0,36 of 101,38 of 82,36 of 101,38 of 82,18 of 75,25 of 63,11 of 19,3 of 7,...,2,3,0,0,0,0,0,0,0,6
4,0,0,73 of 209,83 of 185,72 of 208,82 of 184,42 of 168,43 of 143,17 of 22,19 of 20,...,2,8,0,0,0,0,0,1,0,3


In [28]:
    # Deleting columns that describe strikes and takedowns before separation:
    
for ColName in df_StrikeList[2:20]:
    del df[ColName]
df.head()

IndentationError: expected an indented block after 'for' statement on line 3 (2377624342.py, line 4)

In [22]:
# The method describes how the fight result was obtained ( Was the fight result a KO/TKO? A submission? (automatic win)
# Or, was the fight result decided by the judges (non-KO/TKO result)? And was that decision unanimous or split?

# This column gives more details on the fight result but do not affect the winrate. Since the focus of this analysis
# is winrate, method values can be discarded.
# In the futur, it could be intresting to see the correltation between the method by which a fight was won/loss and 
# the other component of this dataframe.

del df['method'] 

In [ ]:
# Converting 'time', 'control_time_1' and 'control_time_2' values from minutes (xx:xx) to secondes for an easier anlaysis.

import datetime
import time

df['time'] = pd.to_timedelta('00:'+df['time'])
df['time'] = pd.to_timedelta(df['time']).dt.total_seconds() 

df['control_time_1'] = pd.to_timedelta('00:'+df['control_time_1'])
df['control_time_1'] = pd.to_timedelta(df['control_time_1']).dt.total_seconds() 

df['control_time_2'] = pd.to_timedelta('00:'+df['control_time_2'])
df['control_time_2'] = pd.to_timedelta(df['control_time_2']).dt.total_seconds() 

print(df['time'])
print(df['control_time_1'])
print(df['control_time_2'])

In [ ]:
# Printing a sorted correlation matrix between the 'result' target variable and all the others.

# Correlation values show that knockdowns, ground strikes and control time are the principal components that 
# influence the result of a fight.

# Knockdowns and controle time are logicly understanble variables to assume a correlation with the result of a fight.
#    Knockdowns are events that happen when a fighter partially or fully losses conscisness due to blow(s) to the head. 
#        This causes 

corr_matrix = df.corr()['result']
corr_matrix.sort_values()

# accurcy coorelation : divide landed by attempted and create new column call accurcy  For each strike column  - ur hypothsis is only for sign. strikes


In [ ]:
# Here's a heat map of the correlation that is shared between each column. This will help to have an overview of how variables
# interact with each other and could lead to new study paths.

import seaborn as sns
sns.set_style('dark', {'axes.grid' : False})

plt.figure(figsize=(12, 10))
sns.heatmap(data=df.corr(), cmap='coolwarm', center=0, mask=np.triu(df.corr()))


In [ ]:
should i do anoth table for the strike accurcy?

In [ ]:
PCA   df.iloc[f'{n1},{n2}']

In [ ]:
 df% =
    For values in df.iloc[f'{n}']:
        {
...     'Algothime': [f'{df.iloc[0]}', '],
...     'Param. 1': [f'-', '-', '-'],
...     'Param. 2': ['-', '-', '-'],
...     'Plage param. 1': ['-', '-', '-'],
...     'Plage param. 2': ['-', '-', '-'],
... }
                      
                      
    

In [ ]:
df.iloc[0,2]

In [ ]:
for row in df.iterrows():
    print(row)